# Notebook 2: 01_generate_mock_data
Generate device, event, alert, case, and threshold experiment tables.
Export both CSV and SQLite.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path


In [ ]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
FIG_DIR = PROJECT_ROOT / 'figures'
DB_PATH = DATA_DIR / 'fireworks_monitoring.db'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CONFUSION = {'TP':2418,'FN':376,'FP':534,'TN':5874}
NEG_TOTAL = {'truck_horn':1700,'piling':1300,'vehicle_backfire':900,'metal_hit':700,'crowd_noise':700,'machinery':600,'wind_rain':508}
FP_TARGET = {'truck_horn':205,'piling':158,'vehicle_backfire':72,'metal_hit':46,'crowd_noise':31,'machinery':15,'wind_rain':7}
THRESHOLDS = {'v060':0.60,'v070':0.70,'v080':0.80,'v085':0.85,'v090':0.90}
SCENES = ['residential_area','arterial_road','construction_site','industrial_park']


In [ ]:
def sample_scores(cls, n, rng):
    if cls == 'firework':
        n1 = int(n * 0.8)
        return np.concatenate([rng.beta(8,2,n1), rng.beta(4,3,n-n1)])
    params = {
        'truck_horn': (4.5, 3.5), 'piling': (4.2, 3.8), 'vehicle_backfire': (3.5, 4.2),
        'metal_hit': (2.6, 4.8), 'crowd_noise': (2.4, 5.0), 'machinery': (2.2, 5.5), 'wind_rain': (2.0, 6.0),
    }
    a, b = params.get(cls, (2.0, 6.0))
    return rng.beta(a, b, n)

def enforce_pred_count(df, mask, target, thr, rng):
    idx = df.index[mask]
    s = df.loc[idx, 'model_score']
    p = (s >= thr)
    diff = int(p.sum()) - target
    if diff > 0:
        mv = s[p].sort_values().index[:diff]
        df.loc[mv, 'model_score'] = thr - rng.uniform(0.001, 0.03, len(mv))
    elif diff < 0:
        need = -diff
        mv = s[~p].sort_values(ascending=False).index[:need]
        df.loc[mv, 'model_score'] = thr + rng.uniform(0.001, 0.03, len(mv))

def pick_bucket(cls, rng):
    probs = {
        'firework':[0.12,0.33,0.38,0.17], 'truck_horn':[0.15,0.30,0.40,0.15],
        'piling':[0.60,0.22,0.10,0.08], 'machinery':[0.55,0.22,0.13,0.10], 'wind_rain':[0.25,0.25,0.25,0.25],
    }
    p = probs.get(cls, [0.30,0.30,0.25,0.15])
    return rng.choice(['daytime','evening_peak','night','early_morning'], p=p)

def time_from_bucket(day, b, rng):
    if b == 'daytime': h = int(rng.integers(8,17))
    elif b == 'evening_peak': h = int(rng.integers(17,21))
    elif b == 'night': h = int(rng.integers(21,24))
    else: h = int(rng.integers(0,8))
    return pd.Timestamp(day) + pd.Timedelta(hours=h, minutes=int(rng.integers(0,60)), seconds=int(rng.integers(0,60)))

def gen_device_info(rng, n=120):
    scene = rng.choice(SCENES, n, p=[0.35,0.25,0.25,0.15])
    return pd.DataFrame({
        'device_id':[f'D{i:04d}' for i in range(1,n+1)],
        'region_id':rng.choice([f'R{i:02d}' for i in range(1,9)], n),
        'scene_type':scene,
        'install_type':rng.choice(['pole','building_wall','roof'], n, p=[0.60,0.30,0.10]),
        'latitude':31.20 + rng.normal(0,0.06,n),
        'longitude':121.47 + rng.normal(0,0.08,n),
        'firmware_version':rng.choice(['fw_1.7.2','fw_1.8.0','fw_1.8.3'], n, p=[0.2,0.5,0.3]),
        'threshold_version':rng.choice(['v080','v085'], n, p=[0.75,0.25]),
        'mic_array_type':rng.choice(['4mic_linear','6mic_circular'], n, p=[0.65,0.35]),
        'status':rng.choice(['active','maintenance'], n, p=[0.96,0.04]),
    })

def gen_audio_base(device_df, rng):
    rows = [{'binary_label':1,'true_class':'firework'} for _ in range(2418+376)]
    for c,n in NEG_TOTAL.items(): rows += [{'binary_label':0,'true_class':c} for _ in range(n)]
    df = pd.DataFrame(rows)

    scene_prob = {
        'firework':[0.42,0.25,0.15,0.18], 'truck_horn':[0.18,0.48,0.10,0.24], 'piling':[0.08,0.12,0.65,0.15],
        'vehicle_backfire':[0.20,0.40,0.15,0.25], 'metal_hit':[0.18,0.22,0.35,0.25], 'crowd_noise':[0.40,0.25,0.15,0.20],
        'machinery':[0.10,0.15,0.45,0.30], 'wind_rain':[0.30,0.25,0.20,0.25],
    }
    scene_to_dev = {s:device_df.loc[device_df.scene_type==s,'device_id'].tolist() for s in SCENES}
    dev_region = dict(zip(device_df.device_id, device_df.region_id))

    scenes, devs, regions = [], [], []
    for c in df.true_class.tolist():
        s = rng.choice(SCENES, p=scene_prob[c]); pool = scene_to_dev.get(s) or device_df.device_id.tolist(); d = rng.choice(pool)
        scenes.append(s); devs.append(d); regions.append(dev_region[d])
    df['scene_type']=scenes; df['device_id']=devs; df['region_id']=regions

    days = [pd.Timestamp('2026-01-01') + pd.Timedelta(days=i) for i in range(30)]
    festivals = {pd.Timestamp('2026-01-05'),pd.Timestamp('2026-01-16'),pd.Timestamp('2026-01-26')}
    ts,bucket,holiday = [],[],[]
    for c in df.true_class.tolist():
        if c == 'firework':
            w=[]
            for d in days:
                v=1.0 + (4.0 if d in festivals else 0.0) + (1.5 if d.weekday()>=5 else 0.0)
                w.append(v)
            day = rng.choice(days, p=np.array(w)/np.sum(w))
        else:
            day = rng.choice(days)
        b = pick_bucket(c, rng)
        ts.append(time_from_bucket(day,b,rng)); bucket.append(b); holiday.append(int((day.weekday()>=5) or (day in festivals)))
    df['event_time']=pd.to_datetime(ts); df['time_bucket']=bucket; df['is_holiday']=holiday
    df['weather']=rng.choice(['clear','cloudy','light_rain','heavy_rain','windy'], len(df), p=[0.34,0.30,0.18,0.08,0.10])

    ranges = {
        'firework':      {'snr_db':(-5,38),'peak_db':(95,125),'duration_ms':(200,1200),'spectral_centroid':(2200,5600),'zero_crossing_rate':(0.06,0.30),'rms_energy':(0.30,0.95)},
        'truck_horn':    {'snr_db':(-3,32),'peak_db':(90,115),'duration_ms':(500,1800),'spectral_centroid':(1800,4800),'zero_crossing_rate':(0.05,0.24),'rms_energy':(0.22,0.92)},
        'piling':        {'snr_db':(-4,28),'peak_db':(88,118),'duration_ms':(700,2000),'spectral_centroid':(1400,4200),'zero_crossing_rate':(0.04,0.22),'rms_energy':(0.20,0.88)},
        'vehicle_backfire': {'snr_db':(-6,26),'peak_db':(70,116),'duration_ms':(120,1200),'spectral_centroid':(1200,5000),'zero_crossing_rate':(0.03,0.24),'rms_energy':(0.15,0.86)},
        'metal_hit':     {'snr_db':(-5,24),'peak_db':(60,110),'duration_ms':(80,1000),'spectral_centroid':(1000,4700),'zero_crossing_rate':(0.02,0.22),'rms_energy':(0.12,0.80)},
        'crowd_noise':   {'snr_db':(-6,22),'peak_db':(55,100),'duration_ms':(600,2800),'spectral_centroid':(900,3600),'zero_crossing_rate':(0.02,0.17),'rms_energy':(0.08,0.70)},
        'machinery':     {'snr_db':(-5,24),'peak_db':(58,108),'duration_ms':(500,2600),'spectral_centroid':(1000,3800),'zero_crossing_rate':(0.02,0.18),'rms_energy':(0.10,0.75)},
        'wind_rain':     {'snr_db':(-8,18),'peak_db':(40,75),'duration_ms':(900,3200),'spectral_centroid':(600,2800),'zero_crossing_rate':(0.01,0.13),'rms_energy':(0.05,0.60)},
    }

    for c, rg in ranges.items():
        m = (df.true_class==c); n=int(m.sum())
        for k,(lo,hi) in rg.items(): df.loc[m,k] = rng.uniform(lo,hi,n)
        s = sample_scores(c,n,rng); rng.shuffle(s); df.loc[m,'model_score']=s

    thr = 0.80
    enforce_pred_count(df, df.binary_label==1, 2418, thr, rng)
    for c,t in FP_TARGET.items(): enforce_pred_count(df, (df.binary_label==0) & (df.true_class==c), t, thr, rng)
    df['predicted_label'] = (df.model_score >= thr).astype(int)
    df['error_type'] = np.select([
        (df.binary_label==1)&(df.predicted_label==1),
        (df.binary_label==1)&(df.predicted_label==0),
        (df.binary_label==0)&(df.predicted_label==1)
    ], ['TP','FN','FP'], default='TN')

    miss = np.array([None]*len(df), dtype=object)
    fn_idx = df.index[df.error_type=='FN']
    miss[fn_idx] = rng.choice(['far_distance_low_snr','occlusion_reflection','small_fragmented_blast','rain_or_wind_masking'], len(fn_idx), p=[0.40,0.25,0.20,0.15])
    df['miss_reason'] = miss

    df = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    df['clip_id'] = [f'C{i:07d}' for i in range(1, len(df)+1)]

    cm = df.error_type.value_counts().to_dict()
    assert cm.get('TP',0)==2418 and cm.get('FN',0)==376 and cm.get('FP',0)==534 and cm.get('TN',0)==5874

    cols = ['clip_id','event_time','device_id','region_id','scene_type','time_bucket','is_holiday','weather','snr_db','peak_db','duration_ms','spectral_centroid','zero_crossing_rate','rms_energy','true_class','binary_label','model_score','predicted_label','error_type','miss_reason']
    return df[cols]


In [ ]:
def inject_quality_issues(df, rng):
    d = df.copy()
    d.loc[rng.choice(d.index, 130, replace=False), 'weather'] = np.nan
    aliases = {
        'residential_area':['residential','residental_area'], 'arterial_road':['main_road','arterialRoad'],
        'construction_site':['construction','site_construction'], 'industrial_park':['industrialPark','industry_park']
    }
    for i in rng.choice(d.index, 220, replace=False):
        s = d.at[i,'scene_type']
        if s in aliases: d.at[i,'scene_type'] = rng.choice(aliases[s])
    dup = d.loc[rng.choice(d.index, 8, replace=False)].copy()
    dup['peak_db'] = rng.uniform(155, 190, len(dup))
    return pd.concat([d, dup], ignore_index=True)

def gen_alert(audio, thresholds, rng):
    blocks = []
    for ver, thr in thresholds.items():
        a = audio.loc[audio.model_score >= thr].copy().reset_index(drop=True)
        if a.empty: continue
        score = a.model_score.to_numpy()
        blocks.append(pd.DataFrame({
            'alert_id':[f'A_{ver}_{i:07d}' for i in range(1, len(a)+1)],
            'clip_id':a.clip_id,
            'device_id':a.device_id,
            'alert_time':a.event_time + pd.to_timedelta(rng.integers(5,180,len(a)), unit='s'),
            'time_bucket':a.time_bucket,
            'threshold_version':ver,
            'score_at_alert':a.model_score,
            'alert_level':np.where(score>=0.90,'high',np.where(score>=0.82,'medium','low')),
            'pushed_to_platform':rng.choice([1,0], len(a), p=[0.97,0.03]),
            'pushed_delay_ms':rng.integers(250,6000,len(a)),
            'operator_review_result':np.where(a.binary_label.to_numpy()==1,'confirmed_firework','false_alarm'),
            'review_time_sec':np.where(a.binary_label.to_numpy()==1, rng.integers(40,320,len(a)), rng.integers(25,220,len(a))),
        }))
    return pd.concat(blocks, ignore_index=True)

def gen_case(audio, device, rng):
    fire = audio.loc[audio.binary_label==1, ['event_time','device_id','scene_type']].copy()
    fire['window'] = fire.event_time.dt.floor('30min')
    geo = device.set_index('device_id')[['latitude','longitude']]
    rows = []
    for i,(_,g) in enumerate(fire.groupby('window'), 1):
        lat = geo.loc[g.device_id, 'latitude'].mean(); lon = geo.loc[g.device_id, 'longitude'].mean(); mode_scene = g.scene_type.mode().iloc[0]
        rows.append({
            'case_id':f'CASE{i:06d}',
            'start_time':g.event_time.min()-pd.Timedelta(seconds=60),
            'end_time':g.event_time.max()+pd.Timedelta(seconds=120),
            'involved_device_count':int(max(1,g.device_id.nunique())),
            'fused_latitude':float(lat + rng.normal(0,0.002)),
            'fused_longitude':float(lon + rng.normal(0,0.003)),
            'estimated_scene_type':mode_scene,
            'final_result':'confirmed_firework' if len(g)>=2 else 'suspected_firework',
            'response_status':rng.choice(['handled','closed','under_review'], p=[0.45,0.35,0.20]),
            'response_time_sec':int(rng.integers(120,1800)),
            'evidence_generated':int(rng.choice([1,0], p=[0.90,0.10])),
        })
    return pd.DataFrame(rows)

def threshold_log(audio, thresholds):
    out = []
    for i,(ver,thr) in enumerate(thresholds.items(), 1):
        pred = (audio.model_score >= thr).astype(int); y = audio.binary_label.astype(int)
        tp = int(((y==1)&(pred==1)).sum()); fp = int(((y==0)&(pred==1)).sum()); fn = int(((y==1)&(pred==0)).sum())
        precision = tp/(tp+fp) if (tp+fp) else 0.0; recall = tp/(tp+fn) if (tp+fn) else 0.0; f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0
        out.append({'exp_id':f'EXP{i:04d}','threshold_version':ver,'threshold_value':thr,'eval_date':'2026-02-01','precision':round(precision,4),'recall':round(recall,4),'f1':round(f1,4),'alert_volume':tp+fp,'false_alarm_count':fp,'estimated_review_cost':int((tp+fp)*8 + fp*80 + fn*300),'notes':'precision-priority evaluation in urban fireworks regulation'})
    return pd.DataFrame(out)


In [ ]:
device_info = gen_device_info(rng, 120)
audio_base = gen_audio_base(device_info, rng)
audio_raw = inject_quality_issues(audio_base, rng)
alert_record = gen_alert(audio_base, THRESHOLDS, rng)
firework_event_case = gen_case(audio_base, device_info, rng)
threshold_experiment_log = threshold_log(audio_base, THRESHOLDS)

print('device_info:', device_info.shape)
print('audio_base:', audio_base.shape)
print('audio_raw:', audio_raw.shape)
print('alert_record:', alert_record.shape)
print('firework_event_case:', firework_event_case.shape)
print('threshold_experiment_log:', threshold_experiment_log.shape)
print('Confusion on clean base:')
print(audio_base.error_type.value_counts())


In [ ]:
device_info.to_csv(DATA_DIR/'device_info.csv', index=False)
audio_raw.to_csv(DATA_DIR/'audio_clip_event.csv', index=False)
alert_record.to_csv(DATA_DIR/'alert_record.csv', index=False)
firework_event_case.to_csv(DATA_DIR/'firework_event_case.csv', index=False)
threshold_experiment_log.to_csv(DATA_DIR/'threshold_experiment_log.csv', index=False)

with sqlite3.connect(DB_PATH) as conn:
    device_info.to_sql('device_info', conn, if_exists='replace', index=False)
    audio_raw.to_sql('audio_clip_event', conn, if_exists='replace', index=False)
    alert_record.to_sql('alert_record', conn, if_exists='replace', index=False)
    firework_event_case.to_sql('firework_event_case', conn, if_exists='replace', index=False)
    threshold_experiment_log.to_sql('threshold_experiment_log', conn, if_exists='replace', index=False)

print('CSV + DB export complete')
print('DB path:', DB_PATH)
